In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Sequential computations in JAX: `jax.lax.scan`
Many computations are inherently sequential: each step consumes the result of the one before it. Examples include integrating an equation of motion, advancing a Markov chain or evaluating any recurrence of the form $s_{n+1}=f(s_n,x_n)$.

The natural way to write this is a Python for loop, but inside a JAX transformation that loop is unrolled at trace time: every iteration is baked individually into the computation graph, so a thousand steps become a thousand-times-larger program.
As a result, the computation can become slow to compile, heavy on memory, and with a length that must be fixed before it ever runs.

We now introduce `jax.lax.scan` as a solution to this problem ([Link to the documentation](https://docs.jax.dev/en/latest/_autosummary/jax.lax.scan.html), read it!).
You write the body of a single step as a pure function, and scan marches it along an input sequence, or simply for a fixed number of steps, compiling that body only once into one rolled loop.
Each step threads a carry, the state passed from one iteration to the next, and emits an output that scan stacks into an array: the carry is the running state you fold forward and the stacked outputs are the trace you keep.
In a single primitive it is both a fold and a map. and that same shape underlies cumulative sums, running statistics, recurrent networks, ODE integrators, and samplers alike which we will explore in this notebook.

## Simple example - Cumulative sum
As a first simple illustration, let us implement a cumulative sum function using `scan`.
In terms of serie, for a given vector $u x_n$, its cumulative sum is given by
$$u_{n+1} = u_n + x_n,$$
$$u_0 = x_0.$$

In [ ]:
xs = jnp.array([3., 1., 4., 1., 5., 9.])
@jax.jit
def cumulative_sum(xs, x_init=0.):
    def add(carry, x):
        # Pure function. Only output the sum of the current value with its preceding cumulated one.
        new = carry + x
        return new, new
    total, running = jax.lax.scan(add, x_init, xs)
    return total, running

Let us compare both implementations:

In [ ]:
print(cumulative_sum(xs))
print(jnp.sum(xs), jnp.cumsum(xs))

They're perfectly identical, what a surprise.

## Exercice - Geometric series & compound interest

Let us suppose we invest $u_0$ with a fixed annual interest rate $r$.
At year $n$, the total accumulated value becomes

$$u_{n+1} = (1+r)u_n.$$

As a geometric serie, the total accumulated value at year $n$ can be directly computed as

$$u_n = (1+r)^nu_0.$$

Using `jax.lax.scan`, compute the value of the total accumulated value up to year $n$ and compare it to the direct calculation.

In [ ]:
def compound_interest(x_init, rate, years):
    def step(x, _):
        return (1.+rate)*x, x         # carry evolves; record value before update
    final, history = jax.lax.scan(step, x_init, length=years)
    return final, history

> _**Note:**_ In this example, we set `xs` to `None`. Instead, the loop count is define by `length`. In that case, `xs` is built as a tuple of `None`.

In [ ]:
print(compound_interest(100., 0.02, 10))
print((1+0.02)**9*100)

## Exercice - Euler integrator

Now that we master financial mathematics, let us focus on numerically solving Ordinal Differential Equations of the form

$$\frac{\mathrm{d}f}{\mathrm{d}t} = A(t, f),$$

with $A(t)$ an arbitrary continuous and first order differentiable function.

In [ ]:
def A(t, f):
    return f

def euler_integrate_scan(f, f0, t0, T, h):
    t = jnp.linspace(t0, T, ((T-t0)/h).astype(int))
    def step(y, t_i):
        return y + h * f(t_i, y), y

    return t, jax.lax.scan(step, f0, t)[1]

def euler_integrate_fori(f, f0, t0, T, h):
    t = jnp.linspace(t0, T, ((T-t0)/h).astype(int))


## Convergence study
Now that you have a nice integrator, study its convergence against the analytical solution to the ODE.
Try to use `vmap` to sweep over step sizes. Will it work? Explain why or why not.

In [ ]:

def euler_error(h):
    t, y = euler_integrate(A, 1., 0, 1., h)
    return jnp.max((y-jnp.exp(t))**2)

In [ ]:
hs = jnp.logspace(-5, -1)
errors = jnp.array([euler_error(h) for h in hs])
#errors = jax.vmap(euler_error)(hs)

In [ ]:
plt.plot(hs, errors)
plt.xscale('log')
plt.yscale('log')
plt.show()

In [ ]:
#plt.plot(t, y)
plt.plot(t, y-jnp.exp(t))
plt.show()

In [ ]:
def attractor(r, x0=0.5, burn=600, keep=500):
    def step(x, _): return r*x*(1-x), x
    x, _  = jax.lax.scan(step, x0, None, length=burn)   # sequential: burn transient
    _, xs = jax.lax.scan(step, x,  None, length=keep)   # record the attractor
    return xs
rs = jnp.linspace(2.5, 4.0, 1700)
X = jax.vmap(attractor)(rs)   # parallel sweep over r

R = jnp.broadcast_to(rs[:, None], X.shape).ravel()   # each r repeated `keep` times
Y = X.ravel()


In [ ]:
plt.figure(figsize=(10., 4.), layout='constrained')
plt.plot(R, Y, ',k')
plt.show()